# Notebook 07 — Final Model Comparison
Reads `results/all_models_results.csv` (produced by notebooks 01–06)  
and generates final comparison charts and report table.

**Metrics:** R², MAE, RMSE, MAPE

In [ ]:
import os, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path
warnings.filterwarnings('ignore')

NOTEBOOK_DIR = Path().resolve()
PROJECT_ROOT = NOTEBOOK_DIR.parent
PLOTS_DIR    = str(PROJECT_ROOT / 'plots')
RESULTS_CSV  = str(PROJECT_ROOT / 'results' / 'all_models_results.csv')
os.makedirs(PLOTS_DIR, exist_ok=True)

plt.rcParams.update({'figure.facecolor':'white','axes.facecolor':'white',
                     'axes.grid':True,'grid.alpha':0.3,'font.size':11})

COLOR_MAP = {
    'ARIMA':             '#888780',
    'Linear Regression': '#B4B2A9',
    'Random Forest':     '#1D9E75',
    'XGBoost':           '#D85A30',
    'GRU':               '#534AB7',
    'LSTM':              '#EF9F27',
}
print(f"Results CSV : {RESULTS_CSV}")
print(f"Plots dir   : {PLOTS_DIR}")


## 1. Load Results

In [ ]:
results_df = pd.read_csv(RESULTS_CSV)
results_df = results_df.sort_values('RMSE').reset_index(drop=True)
results_df.index += 1
print("Results loaded:")
print(results_df.to_string())


## 2. Metrics Bar Charts

In [ ]:
models     = results_df['model'].tolist()
bar_colors = [COLOR_MAP.get(m, '#378ADD') for m in models]

metrics = [('R2','R²','higher'),('MAE','MAE (cores)','lower'),
           ('RMSE','RMSE (cores)','lower'),('MAPE','MAPE (%)','lower')]

fig, axes = plt.subplots(1, 4, figsize=(22, 5))
fig.suptitle('Final Model Comparison — R², MAE, RMSE, MAPE', fontsize=14, fontweight='bold')

for ax, (col, label, better) in zip(axes, metrics):
    vals   = results_df[col].values
    bars   = ax.bar(models, vals, color=bar_colors, edgecolor='white', alpha=0.88)
    best_i = vals.argmax() if better=='higher' else vals.argmin()
    bars[best_i].set_edgecolor('#E24B4A'); bars[best_i].set_linewidth(2.5)
    ax.set_title(f'{label}\n({better} is better)', fontsize=10)
    ax.set_ylabel(label); ax.tick_params(axis='x', rotation=35)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x()+bar.get_width()/2,
                bar.get_height()+max(abs(vals))*0.01,
                f'{val:.4f}', ha='center', va='bottom', fontsize=8)

patches = [mpatches.Patch(color=c, label=m) for m,c in COLOR_MAP.items() if m in models]
fig.legend(handles=patches, loc='lower center', ncol=6, bbox_to_anchor=(0.5,-0.08), fontsize=9)
plt.tight_layout()
out = os.path.join(PLOTS_DIR, '07_final_bars.png')
plt.savefig(out, dpi=150, bbox_inches='tight'); plt.show()
print(f"Saved: {out}")


## 3. Composite Score

In [ ]:
df_n = results_df.copy()
for col, higher in [('R2',True),('MAE',False),('RMSE',False),('MAPE',False)]:
    rng = df_n[col].max()-df_n[col].min()+1e-9
    df_n[col+'_n'] = (df_n[col]-df_n[col].min())/rng if higher else 1-(df_n[col]-df_n[col].min())/rng
df_n['score'] = (df_n['R2_n']+df_n['MAE_n']+df_n['RMSE_n']+df_n['MAPE_n'])/4

bar_c  = [COLOR_MAP.get(m,'#378ADD') for m in df_n['model']]
fig, ax = plt.subplots(figsize=(10,5))
bars   = ax.bar(df_n['model'], df_n['score'], color=bar_c, edgecolor='white', alpha=0.88)
best_i = df_n['score'].argmax()
bars[best_i].set_edgecolor('#E24B4A'); bars[best_i].set_linewidth(2.5)
ax.set_title('Composite Score (norm. R², MAE, RMSE, MAPE — higher = better)',
             fontsize=12, fontweight='bold')
ax.set_ylabel('Score (0–1)'); ax.set_ylim(0,1.15); ax.tick_params(axis='x', rotation=30)
for bar, val in zip(bars, df_n['score']):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.01,
            f'{val:.3f}', ha='center', va='bottom', fontsize=9, fontweight='bold')
plt.tight_layout()
out = os.path.join(PLOTS_DIR, '07_composite_score.png')
plt.savefig(out, dpi=150, bbox_inches='tight'); plt.show()
print(f"Saved: {out}")


## 4. Report-Ready Table

In [ ]:
print("\n" + "="*72)
print("  FINAL RESULTS — REPORT TABLE")
print("="*72)
print(f"{'Rank':<6}{'Model':<22}{'R²':>8}{'MAE (cores)':>14}{'RMSE':>10}{'MAPE (%)':>12}")
print("-"*72)
for i, row in results_df.iterrows():
    tag = '  ← BEST' if row['RMSE']==results_df['RMSE'].min() else ''
    print(f"{i:<6}{row['model']:<22}{row['R2']:>8.4f}{row['MAE']:>14.4f}{row['RMSE']:>10.4f}{row['MAPE']:>12.2f}{tag}")
print("="*72)
best = results_df.iloc[0]
print(f"\nBest model : {best['model']}")
print(f"  R²={best['R2']:.4f} | MAE={best['MAE']:.4f} cores | RMSE={best['RMSE']:.4f} | MAPE={best['MAPE']:.2f}%")
print(f"\nResults CSV → {RESULTS_CSV}")
print(f"All plots   → {PLOTS_DIR}/")
